In [1]:
import sys
sys.path.insert(0, '../src') 

import jax 
import jax.numpy as jnp 
import optax 
import numpy as np 
import matplotlib.pyplot as plt 

from optimizers import upgd_optimizer, UPGDState 

KEY = jax.random.PRNGKey(42)

In [2]:
params = {"w": jnp.array([1.0, -2.0, 3.0]), "b": jnp.array([0.5])}

grad = jax.tree.map(lambda p: jnp.ones_like(p) * 2, params)
hess = jax.tree.map(lambda p: jnp.ones_like(p) * 0.1, params)

print(grad)
print(hess)

{'b': Array([2.], dtype=float32), 'w': Array([2., 2., 2.], dtype=float32)}
{'b': Array([0.1], dtype=float32), 'w': Array([0.1, 0.1, 0.1], dtype=float32)}


In [3]:
opt = upgd_optimizer( 
    step_size=1e-2, 
    utility_decay_rate=0.9, 
    noise=0.0, 
    seed=0
)
print("Optimizer:", opt)
state = opt.init(params)
print("State:", state)

Optimizer: GradientTransformationExtraArgs(init=<function upgd_optimizer.<locals>.init_fn at 0x1280ce200>, update=<function upgd_optimizer.<locals>.update_fn at 0x1280cdc60>)
State: UPGDState(step=Array(0, dtype=int32), avg_utility={'b': Array([0.], dtype=float32), 'w': Array([0., 0., 0.], dtype=float32)}, rng_key=Array([0, 0], dtype=uint32))


In [4]:
print(isinstance(state, UPGDState))  # ensures state is of class UPGDState 
print(state.step == 0)               # ensures timestep t starts at 0 
print(jax.tree.map(jnp.shape, state.avg_utility) == jax.tree.map(jnp.shape, params))  # ensures the utility is the same shape as the params network 
print(state.rng_key)

True
True
True
[0 0]


In [5]:
updates, new_state = opt.update( 
    updates = jax.tree.map(jnp.zeros_like, params), 
    state=state,
    params=params,
    grad=grad,
    hess_diag=hess,
)

new_params = optax.apply_updates(params, updates)

print(params)
print(updates)
print(new_state)
print(new_params)

RAW UTILITY: {'b': Array([-0.9875], dtype=float32), 'w': Array([-1.95,  4.2 , -5.55], dtype=float32)}
RUNNING AVG UTILITY: {'b': Array([-0.09875], dtype=float32), 'w': Array([-0.19500001,  0.42      , -0.555     ], dtype=float32)}
U_BAR: {'b': Array([0.44148952], dtype=float32), 'w': Array([0.3859696 , 0.7310586 , 0.21058069], dtype=float32)}
PERTURB: {'b': Array([-0.], dtype=float32), 'w': Array([ 0.,  0., -0.], dtype=float32)}
{'w': Array([ 1., -2.,  3.], dtype=float32), 'b': Array([0.5], dtype=float32)}
{'b': Array([-0.01117021], dtype=float32), 'w': Array([-0.01228061, -0.00537883, -0.01578839], dtype=float32)}
UPGDState(step=Array(1, dtype=int32), avg_utility={'b': Array([-0.09875], dtype=float32), 'w': Array([-0.19500001,  0.42      , -0.555     ], dtype=float32)}, rng_key=Array([1797259609, 2579123966], dtype=uint32))
{'b': Array([0.4888298], dtype=float32), 'w': Array([ 0.9877194, -2.0053787,  2.9842117], dtype=float32)}


In [6]:
print(new_state.avg_utility["w"])

[-0.19500001  0.42       -0.555     ]


In [7]:
state = opt.init(params)
p = params
for t in range(10):
    upd, state = opt.update(
        updates=jax.tree.map(jnp.zeros_like, p),
        state=state, params=p,
        grad=grad, hess_diag=hess,
    )
    p = optax.apply_updates(p, upd)
print("step:", state.step)              # expect 10
print("u_avg:", state.avg_utility["w"]) # expect non-zero, all same sign
print("params:", p["w"])                # expect drift away from initial


RAW UTILITY: {'b': Array([-0.9875], dtype=float32), 'w': Array([-1.95,  4.2 , -5.55], dtype=float32)}
RUNNING AVG UTILITY: {'b': Array([-0.09875], dtype=float32), 'w': Array([-0.19500001,  0.42      , -0.555     ], dtype=float32)}
U_BAR: {'b': Array([0.44148952], dtype=float32), 'w': Array([0.3859696 , 0.7310586 , 0.21058069], dtype=float32)}
PERTURB: {'b': Array([-0.], dtype=float32), 'w': Array([ 0.,  0., -0.], dtype=float32)}
RAW UTILITY: {'b': Array([-0.96571183], dtype=float32), 'w': Array([-1.9266593,  4.2118344, -5.5231476], dtype=float32)}
RUNNING AVG UTILITY: {'b': Array([-0.18544619], dtype=float32), 'w': Array([-0.36816594,  0.7991834 , -1.0518148 ], dtype=float32)}
U_BAR: {'b': Array([0.44224772], dtype=float32), 'w': Array([0.38682505, 0.7310586 , 0.21146588], dtype=float32)}
PERTURB: {'b': Array([-0.], dtype=float32), 'w': Array([-0.,  0., -0.], dtype=float32)}
RAW UTILITY: {'b': Array([-0.9439408], dtype=float32), 'w': Array([-1.9033362,  4.2236724, -5.4962997], dtype=fl